In [ ]:
# Install crew AI
# pip install -U crewai
import os

os.environ["OPENAI_API_KEY"] = "YOUR-OPEN-AI-KEY"  # Replace with your key


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.3/576.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.1/250.1

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

# Define LLM for agents
llm = LLM(model="gpt-3.5-turbo")  # You can use "gpt-4" if desired

# Define agents
planner = Agent(
    role="Travel Planner",
    goal="Decompose the user's flight booking task into meaningful subtasks and route it to the correct agent.",
    backstory="You're a smart travel planner who understands how to break down complex flight booking tasks and direct them appropriately.",
    llm=llm,
    verbose=True
)

domestic_agent = Agent(
    role="Domestic Flight Booker",
    goal="Find the best flight options within the user's home country based on user preferences.",
    backstory="You specialize in domestic flight bookings and are known for efficient and affordable recommendations.",
    llm=llm,
    verbose=True
)

international_agent = Agent(
    role="International Flight Booker",
    goal="Find the best international flights within budget and help the user prepare documents.",
    backstory="You are an expert in global travel logistics and booking cost-effective international flights.",
    llm=llm,
    verbose=True
)

# Define tasks
planner_task = Task(
    description=(
        "Decide whether the destination '{destination}' is domestic or international "
        "and forward the booking task to the right agent. Then summarize what happens."
    ),
    expected_output="An overview of which agent was selected and what steps were taken.",
    agent=planner
)

domestic_task = Task(
    description=(
        "Search for the best domestic flights to '{destination}' from '{departure_city}' "
        "under ${budget} and prepare one option with airline, timing, and price."
    ),
    expected_output="One domestic flight option with airline, time, and cost.",
    agent=domestic_agent
)

international_task = Task(
    description=(
        "Search international flights to '{destination}' from '{departure_city}' under ${budget}, "
        "and verify required travel documents. Provide at least one option."
    ),
    expected_output="One international flight option and document checklist.",
    agent=international_agent
)

# Set up the crew
crew = Crew(
    agents=[planner, domestic_agent, international_agent],
    tasks=[planner_task, domestic_task, international_task],
    process=Process.sequential,  # Optional: you could use Process.hierarchical later
    verbose=True
)

# Provide sample input
inputs = {
    "destination": "Paris",
    "departure_city": "New York",
    "budget": 800
}

# Run the crew
result = crew.kickoff(inputs=inputs)
print("\n✅ Final Output:\n", result.raw)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 90757b6b-4733-411d-9797-7beb17348a48                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planner                                                                                          │
│                                                                                                                 │
│  Task: Decide whether the destination 'Paris' is domestic or international and forward the booking task to the  │
│  right agent. Then summarize what happens.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planner                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I now can give a great answer.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 18fdcc2c-54cf-4132-9b47-ecc0c99908df                                                                     │
│  Agent: Domestic Flight Booker                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: International Flight Booker                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  For the international flight from New York to Paris under $800, one of the best options is with Air France.    │
│  Airline: Air France                                                                                            │
│  Timing: Departure at 9:00 AM, Arrival at 3:00 PM                                                               │
│  Price: $750                                                                                                    │
│                                                                                                                 │
│  Travel Document Checklist:                                                                                     │
│  1. **Passport:** Ensure your passport is valid for at least six months beyond your planned date of return.     │
│  2. **Visa:** Check if you need a visa for France based on your nationality. Apply for a Schengen visa if       │
│  required.                                                                                                      │
│  3. **Flight Itinerary:** Print out your flight itinerary which includes your departure and arrival details.    │
│  4. **Travel Insurance:** Consider getting travel insurance to cover unforeseen circumstances during your       │
│  trip.                                                                                                          │
│  5. **Accommodation Confirmation:** Have confirmation of your stay in Paris.                                    │
│  6. **Proof of Sufficient Funds:** Carry proof of funds to cover your expenses during your stay.                │
│  7. **COVID-19 Documentation:** Check the latest travel restrictions and requirements due to the pandemic.      │
│  Carry necessary documents like vaccination certificates, test results, and health declaration forms.           │
│                                                                                                                 │
│  Ensure you have all these documents in order before your travel date. Safe travels!                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯